In [0]:
dbutils.widgets.text("storage_account_name", "bhaskar")
storage_account_name=dbutils.widgets.get("storage_account_name")
print(storage_account_name)

# Connecting with ADLS Gen2

In [0]:
%run /Workspace/Repos/azbhaskar18@outlook.com/adfrepo-aug/Notebooks/Connectivitity

### Creating database

In [0]:
%sql
create database if not exists hive_metastore.gold_db

In [0]:
%sql
use gold_db

### Reading data from Silver to Gold and applying transformation

In [0]:
#
from pyspark.sql.functions import *
finalres=spark.sql("""
            SELECT * FROM silver_db.calendar
          """)

finalres.write.format("delta").mode("append").save(f"abfss://gold@{storage_account_name}.dfs.core.windows.net/calendar/")

In [0]:
spark.sql(f"""
create table if not exists gold_db.calendar (
    Date date,
    MonthNbr int,
    Month string,
    Year int
)
location 'abfss://gold@{storage_account_name}.dfs.core.windows.net/calendar'
""")

#### dim_product_details table

In [0]:
spark.sql(f"""
create table if not exists gold_db.dim_product_details (
    product_key int,
    product string,
    subcategory string,
    category string
)
location 'abfss://gold@{storage_account_name}.dfs.core.windows.net/dim_product_details'
""")

In [0]:
df1=spark.sql("""
SELECT 
  p.ProductKey AS product_key,
  p.Product AS product,
  s.Subcategory AS subcategory,
  c.Category AS category
 FROM silver_db.product AS p 
 INNER JOIN silver_db.subcategory AS s  ON p.SubcategoryKey = s.SubcategoryKey
 INNER JOIN silver_db.category AS c ON s.categorykey=c.categorykey
 """)


In [0]:
df1.write.format("delta").mode("append").save(f"abfss://gold@{storage_account_name}.dfs.core.windows.net/dim_product_details/")

In [0]:
spark.sql(f"""
create table if not exists gold_db.fact_sales (
    order_date date,
    product_key int,
    order_qty int
)
location 'abfss://gold@{storage_account_name}.dfs.core.windows.net/fact_sales'
""")

DataFrame[]

In [0]:
df1=spark.sql("""
    SELECT 
    OrderDate AS order_date, 
    ProductKey AS product_key, 
    CAST(SUM(OrderQuantity) AS INT) AS order_qty
    FROM silver_db.sales
    GROUP BY 
    OrderDate,
    ProductKey
"""
)
#df1.printSchema()
df1.write.format("delta").mode("append").save(f"abfss://gold@{storage_account_name}.dfs.core.windows.net/fact_sales/")

In [0]:
%sql
SELECT * FROM gold_db.fact_sales

order_date,product_key,order_qty
2017-01-01,480,111
2017-01-11,487,20
2017-01-12,574,4
2017-01-16,536,29
2017-01-01,381,6
2017-01-09,569,1
2017-01-12,471,6
2017-01-10,541,32
2017-01-30,226,6
2017-01-31,214,20
